# 2.4 基于 Add 算子的完整开发

## 概述

上一节已经运行了一维向量加法。这一节把它扩展成一次完整的 Add 开发：先明确输入、输出和支持条件，再把 `N=2048` 的数据分给两个逻辑块，最后对照 JIT 和显式编译两种入口，分别调用 kernel，并与 PyTorch 参考结果比较。

## 学习目标

- 根据公式明确输入、输出和支持条件。
- 计算每个逻辑块负责的数据范围。
- 写出包含数据搬运和向量计算的完整 Add kernel。
- 使用 `@tilelang.jit` 编译并调用 kernel，使用参考结果验证完整输出。
- 了解 `tilelang.compile` 这一显式编译入口。

## 1. 先明确 Add 要做什么

写代码前先确定算子的对外行为。本节使用下面这组固定参数作为开发目标：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">项目</th>
<th style="text-align: left; vertical-align: top;">本节约定</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">计算公式</td>
<td style="text-align: left; vertical-align: top;"><code>c[i] = a[i] + b[i]</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">输入</td>
<td style="text-align: left; vertical-align: top;"><code>a, b: (2048,) float32</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">输出</td>
<td style="text-align: left; vertical-align: top;"><code>c: (2048,) float32</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">逻辑任务</td>
<td style="text-align: left; vertical-align: top;"><code>num_blocks = 2</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">每块数据量</td>
<td style="text-align: left; vertical-align: top;"><code>tile = 2048 / 2 = 1024</code> 个元素</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">当前限制</td>
<td style="text-align: left; vertical-align: top;"><code>N</code> 能被 <code>num_blocks</code> 整除，且 <code>tile</code> 是 64 的倍数</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">验证方式</td>
<td style="text-align: left; vertical-align: top;">与 <code>a + b</code> 逐元素比较，<code>rtol=0</code>、<code>atol=0</code></td>
</tr>
</tbody>
</table>

这里的输出由 kernel 另行写入，输入 `a` 和 `b` 保持不变。当前代码每次处理完整的 64-lane float32 向量，因此暂不处理不足 64 个元素的尾块。

## 2. 把数据分给两个逻辑块

每个逻辑块通过自己的编号 `bx` 计算起始位置：

```python
tile = n // num_blocks
begin = bx * tile
end = begin + tile
```

代入本节参数后，各块负责的范围是：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;"><code>bx</code></th>
<th style="text-align: left; vertical-align: top;"><code>begin</code></th>
<th style="text-align: left; vertical-align: top;"><code>end</code></th>
<th style="text-align: left; vertical-align: top;">处理范围</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">1024</td>
<td style="text-align: left; vertical-align: top;"><code>[0, 1024)</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">1024</td>
<td style="text-align: left; vertical-align: top;">2048</td>
<td style="text-align: left; vertical-align: top;"><code>[1024, 2048)</code></td>
</tr>
</tbody>
</table>

两个区间首尾相接、没有重叠，也没有漏掉元素。这里的 `num_blocks=2` 表示两份逻辑任务，两个任务最终由几个物理核执行，由编译器和运行时调度。它们可能被分配到两个物理核并行执行，也可能在一个物理核中执行。

![两个逻辑块各自处理 Add 的输入和输出切片](images/02.04_add_workflow_v2.png)

图注：A、B、C 各有 2048 个 float32 元素。两个逻辑块运行同一份 kernel 代码，但依据 `bx` 取得各自的 1024 元素区间；每块再分 16 轮、每轮处理 64 个元素。两个输出区间拼接后正好覆盖 `[0,2048)`。这里画的是逻辑任务，不代表固定使用两个物理核。

## 3. 按数据通路写出 Add kernel

与上一节的一维加法相比，这里最主要的变化是两个逻辑块通过 `bx` 处理不同的数据范围；CopyIn → Compute → CopyOut 的数据通路保持不变。阅读下面的代码时，可以依次找到五个位置：数据接口、当前块范围、UB 工作区、向量计算和结果写回。

```python
def add_program(n: int, num_blocks: int):
    tile = n // num_blocks

    @T.prim_func
    def main(
        a: T.Tensor((n,), T.float32),
        b: T.Tensor((n,), T.float32),
        c: T.Tensor((n,), T.float32),
    ):
        with T.Kernel(num_blocks) as bx:
            # 1. 当前逻辑块负责的起始位置
            begin = bx * tile

            # 2. 当前 tile 在 UB 中的工作区
            a_ub = T.alloc_shared((tile,), T.float32)
            b_ub = T.alloc_shared((tile,), T.float32)
            c_ub = T.alloc_shared((tile,), T.float32)

            # 3. CopyIn：GM → UB
            T.copy(a[begin : begin + tile], a_ub)
            T.copy(b[begin : begin + tile], b_ub)

            # 4. Compute：UB → 寄存器 → UB
            with T.SimdVF():
                mask = T.simd.pset(32)
                for k in T.serial(tile // 64):
                    a_vec = T.simd.vld(a_ub[k * 64])
                    b_vec = T.simd.vld(b_ub[k * 64])
                    c_vec = T.simd.vadd(a_vec, b_vec, mask)
                    T.simd.vsts(c_ub[k * 64], c_vec, mask)

            # 5. CopyOut：UB → GM
            T.copy(c_ub, c[begin : begin + tile])

    return main
```

这段代码使用 `T.Tensor` 声明输入输出。第 1 章出现过的 `T.Buffer` 在简单示例中也承担 shape 和 dtype 接口声明的作用；本节不展开二者的底层差异。

## 4. 编译、调用并验证

写好 `add_program` 后，Host 可以通过两种入口取得可调用的 kernel：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">入口</th>
<th style="text-align: left; vertical-align: top;">怎么使用</th>
<th style="text-align: left; vertical-align: top;">本节做什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>@tilelang.jit</code></td>
<td style="text-align: left; vertical-align: top;">装饰 <code>make_add</code>，调用工厂函数时根据 <code>n</code> 和 <code>num_blocks</code> 得到 kernel</td>
<td style="text-align: left; vertical-align: top;">编译并调用 Add</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>tilelang.compile</code></td>
<td style="text-align: left; vertical-align: top;">把 <code>add_program(n, num_blocks)</code> 显式交给编译器</td>
<td style="text-align: left; vertical-align: top;">调用同一份 Add，并查看生成源码</td>
</tr>
</tbody>
</table>

两种入口中的 `out_idx=-1` 都表示参数列表最后一个 Tensor `c` 是输出，所以调用时只传入 `a` 和 `b`。下面的完整脚本使用同一组输入分别运行两种入口，并把**两个完整输出**都与 `a + b` 比较；还会筛选显示生成源码中与搬运和向量计算有关的行。

课后练习可以只把 `run()` 中的 `num_blocks` 从 2 改为 4，检查完整输出是否仍然正确。

<p><small>先运行 %%writefile 单元保存脚本，再运行后面的执行单元。执行单元在独立进程中编译、调用并比较结果，失败时会报错。</small></p>

<small>线上没有950设备时，可直接阅读本页结果；自行运行请按1.4准备环境，并从下方准备单元开始顺序执行。辅助代码仅负责启动程序与保存日志。</small>

In [1]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile


def run_example(script):
    """在独立进程中运行，并保留完整日志。"""
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    log_dir = Path(os.environ.get("COURSE_OUTPUT_DIR", tempfile.gettempdir())) / "notebook_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_dir.joinpath(Path(script).stem + ".log").write_text(
        result.stdout + result.stderr, encoding="utf-8"
    )
    # 页内显示程序结果；安装路径、编译过程和底层警告保留在原始日志中。
    print("\n".join(
        line for line in result.stdout.splitlines()
        if not line.startswith("Loading tilelang libs from dev root:")
        and not re.match(r"^\d{4}-\d{2}-\d{2}.*\[TileLang:", line)
    ))
    if result.returncode:
        print(result.stderr)
        result.check_returncode()


In [2]:
%%writefile /tmp/tilelang_ch02_add_workflow.py
"""2.4 Add：同一份 kernel 分别通过 JIT 和显式编译运行。"""
import torch
import tilelang
import tilelang.ascend.language as T


def _validate(n: int, num_blocks: int) -> int:
    if n <= 0 or num_blocks <= 0 or n % num_blocks:
        raise ValueError("n、num_blocks 必须为正数，且 n 必须能被 num_blocks 整除")
    tile = n // num_blocks
    if tile % 64:
        raise ValueError(f"tile={tile}，必须是 64 的倍数")
    return tile


def add_program(n: int, num_blocks: int):
    tile = _validate(n, num_blocks)

    @T.prim_func
    def main(
        a: T.Tensor((n,), T.float32),
        b: T.Tensor((n,), T.float32),
        c: T.Tensor((n,), T.float32),
    ):
        with T.Kernel(num_blocks) as bx:
            # 每个逻辑块都有自己的 UB 工作区，只处理自己的 tile。
            a_ub = T.alloc_shared((tile,), T.float32)
            b_ub = T.alloc_shared((tile,), T.float32)
            c_ub = T.alloc_shared((tile,), T.float32)
            begin = bx * tile

            # CopyIn：把当前块负责的两个输入切片搬到 UB。
            T.copy(a[begin : begin + tile], a_ub)
            T.copy(b[begin : begin + tile], b_ub)

            # Compute：每轮从 UB 读取 64 个 float32，在寄存器中相加。
            with T.SimdVF():
                mask = T.simd.pset(32)
                for i in T.serial(tile // 64):
                    a_vec = T.simd.vld(a_ub[i * 64])
                    b_vec = T.simd.vld(b_ub[i * 64])
                    c_vec = T.simd.vadd(a_vec, b_vec, mask)
                    T.simd.vsts(c_ub[i * 64], c_vec, mask)

            # CopyOut：只把当前块的结果写回 C 的对应区间。
            T.copy(c_ub, c[begin : begin + tile])

    return main


@tilelang.jit(out_idx=-1, target="ascend")
def make_add(n: int, num_blocks: int = 1):
    """根据 shape 和逻辑块数量构造可调用的 Add Kernel。"""
    return add_program(n, num_blocks)


def run() -> None:
    # 练习时只需修改这里的 num_blocks，再按顺序重跑 Notebook 两格。
    n, num_blocks = 2048, 2
    _validate(n, num_blocks)
    torch.manual_seed(31)
    a = torch.randn(n, device="npu", dtype=torch.float32)
    b = torch.randn(n, device="npu", dtype=torch.float32)

    # 入口一：调用被 @tilelang.jit 装饰的工厂函数。
    jit_kernel = make_add(n, num_blocks)
    out_jit = jit_kernel(a, b)
    torch.npu.synchronize()
    torch.testing.assert_close(out_jit, a + b, rtol=0.0, atol=0.0)

    # 入口二：显式编译同一份 add_program。
    compiled = tilelang.compile(add_program(n, num_blocks), target="ascend", out_idx=-1)
    out_compile = compiled(a, b)
    torch.npu.synchronize()
    torch.testing.assert_close(out_compile, a + b, rtol=0.0, atol=0.0)

    source = compiled.get_kernel_source()
    print("JIT 和 compile 的输出均通过校验；下面是生成源码中的关键行：")
    for line in source.splitlines():
        if "copy_" in line or "simd_vf" in line or "vadd" in line or "vsts" in line:
            print(line.strip())
    print("Verification passed!")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_add_workflow.py


In [3]:
from pathlib import Path
import subprocess
import sys

# 修改上方代码后，先重跑写入单元，再运行本单元。
script = Path("/tmp/tilelang_ch02_add_workflow.py")
if not script.is_file():
    raise FileNotFoundError("请先运行上方代码单元，生成 Add 脚本")
run_example(str(script))


JIT 和 compile 的输出均通过校验；下面是生成源码中的关键行：
__simd_vf__ inline void main_kernel_simd_vf_0(__ubuf__ uint8_t* buf_dyn_shmem) {
auto c_vec = simd_inst::vadd(a_vec, b_vec, mask, MODE_ZEROING);
simd_inst::vsts(c_vec, (__ubuf__ float*)(&(((__ubuf__ float*)buf_dyn_shmem)[((i * 64) + 2048)])), 0, NORM_B32, mask);
copy_gm_to_ubuf_align_v2((__ubuf__ uint8_t*)((&(((__ubuf__ float*)buf_dyn_shmem)[0]))), (__gm__ uint8_t*)((&(a[(((int32_t)blockIdx.x) * 1024)]))), 0, 1, 4096, 0, 0, 0, 0, 4096, 4096);
copy_gm_to_ubuf_align_v2((__ubuf__ uint8_t*)((&(((__ubuf__ float*)buf_dyn_shmem)[1024]))), (__gm__ uint8_t*)((&(b[(((int32_t)blockIdx.x) * 1024)]))), 0, 1, 4096, 0, 0, 0, 0, 4096, 4096);
main_kernel_simd_vf_0(buf_dyn_shmem);
copy_ubuf_to_gm_align_v2((__gm__ void*)((&(c[(((int32_t)blockIdx.x) * 1024)]))), (__ubuf__ void*)((&(((__ubuf__ float*)buf_dyn_shmem)[2048]))), 0, 1, 4096, 4, 4096, 4096);
Verification passed!


## 小结

本节从 `c[i]=a[i]+b[i]` 出发，先确定输入输出和限制，再用 `bx` 将 2048 个元素分成两个连续区间，最后按照 CopyIn → Compute → CopyOut 写出完整 Add。Host 通过 JIT 取得 kernel、传入数据并比较完整输出；需要直接查看生成源码时，也可以使用 `tilelang.compile`。

## 课后练习

1. 不运行代码，先写出 `N=2048、num_blocks=4` 时每个逻辑块的 `begin`、`end` 和处理范围。
2. 将脚本中的 `num_blocks` 改为 4，运行后确认完整输出仍然通过比较。
3. 尝试 `num_blocks=64`，根据 `tile` 的大小解释为什么 Host 会拒绝这个配置。
4. 说明 `out_idx=-1` 的作用，并比较 `@tilelang.jit` 与 `tilelang.compile` 在本节中的用途。

答案与解析见 [02.04_questions.txt](answer/02.04_questions.txt)，完整脚本见 [02.04_add_workflow.py](answer/02.04_add_workflow.py)。